# By Sage Fox and Nicholas Fox

In [1]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/Python Code/Paper_condensed/scripts/wrtds_python

In [2]:
import os
import pickle
import numpy as np
import pandas as pd
from scipy import stats
import xarray as xr
import multiprocessing as mp
import queue
from pathlib import Path

import rpy2
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri

%load_ext rpy2.ipython

In [3]:
trim_to_water_year=True
alternate_percentile=False
add_fluxbiases=False
start_date, end_date = "2022-4-1", "2023-3-31"
water_year = (pd.to_datetime(start_date), pd.to_datetime(end_date))
nboot = 20
nkalman = 25

data_path = Path.cwd().parent.parent/"data"
results_path = Path.cwd().parent.parent/"autogenerated_results"/"wrtds"

csv_out_root = results_path/"csv"
fig_out_root = results_path/"fig"
netcdf_out_root = results_path/"netcdf"

In [4]:
# Load all the data
from data_prep2 import *
areas = pd.Series(load_areas(data_path/"watershed_area_stats.csv")) # TODO: Make load_areas output a series
#areas_l = [areas[r] for r in rivers]
chem_dfs, detection_limits, rng_df = load_chem_dfs(data_path/"wq_uncen_all.xlsx")
scaflow = load_flow_df(data_path/"Downstream_Estimate_All_Rivers_rolling60.csv", start_date, end_date)
unscaflow = load_flow_df(data_path/"Copy_of_franken_flow.csv", start_date, end_date)

In [5]:
# Fix Cisnes, which is the only one with duplicate dates but does not have b.d. values
Cisnes_chem = chem_dfs["Cisnes"]
Cisnes_chem = Cisnes_chem.groupby(Cisnes_chem.index).mean(numeric_only=True)
Cisnes_chem["Date"] = Cisnes_chem.index.astype("str")
chem_dfs["Cisnes"] = Cisnes_chem

In [6]:
assert chemicals == ["TN", "TP", "Fe", "dSi", "TSS"]
assert rivers == ["Puelo", "Yelcho", "Palena", "Cisnes", "Aysen"]

In [7]:
# %R install.packages(c("EGRET", "lubridate", "zoo", "EGRETci", "dataRetrieval", "spam", "fields"))
if IN_COLAB:
    %R .libPaths("library_folder")

In [8]:
%%R
library(EGRET)
library(lubridate)
library(zoo)
library(EGRETci)
library(dataRetrieval)
library(spam)
library(fields)

source("modded_Rfuncs.R")
source("wrtds_process.R")
# package_names = ('here', 'tidyverse', 'stringr', 'lubridate', 'reshape2', 'data.table',
#                  'dataRetrieval', 'EGRET', 'EGRETci', 'zoo', 'glue', 'hash')
# package_names = ('here', 'reshape2', 'dataRetrieval', 'EGRET', 'EGRETci', 'zoo', 'hash')

EGRET 3.0.11
Extended Documentation: https://doi-usgs.github.io/EGRET

Attaching package: 'lubridate'

The following objects are masked from 'package:base':

    date, intersect, setdiff, union


Attaching package: 'zoo'

The following objects are masked from 'package:base':

    as.Date, as.Date.numeric

EGRETci 2.0.5
Extended Documentation: https://doi-usgs.github.io/EGRETci/
dataRetrieval 2.7.23
Extended Documentation: https://doi-usgs.github.io/dataRetrieval
Learn about the new functions that are replacing NWIS functions here:
https://doi-usgs.github.io/dataRetrieval/articles/read_waterdata_functions.html
Consider adding an API_USGS_PAT for new USGS functions.
See: https://api.waterdata.usgs.gov/signup
Spam version 2.11-3 (2026-01-05) is loaded.
Type 'help( Spam)' or 'demo( spam)' for a short introduction 
and overview of this package.
Help for individual functions is also obtained by adding the
suffix '.spam' to the function name, e.g. 'help( chol.spam)'.

Attaching package: 'spam

In [9]:
create_eList = ro.r["create_eList"]
estimate_tables = ro.r["estimate_tables"]

def pd2R(obj):    
    with (ro.default_converter + pandas2ri.converter).context():
        obj_r = ro.conversion.py2rpy(obj)
    return obj_r
def R2pd(r_obj):
    with (ro.default_converter + pandas2ri.converter).context():
        obj_py = ro.conversion.rpy2py(r_obj)
    return obj_py

In [10]:
def repack_tables(Rlist):
  assert list(Rlist.names) == ['eList', 'dailyBootOut', 'dayPct', 'ContConc', 'error'] # sanity check in case things get moved around... hopefully not
  repacked = {
      "eList_INFO": R2pd(Rlist[0][0]),
      "eList_Daily": R2pd(Rlist[0][1]),
      "eList_Sample": R2pd(Rlist[0][2]),
      "eList_surfaces": Rlist[0][3], # Already an ndarray
      # TODO: Remove the eList part to save time? Remove it from wrtds_process?

      "dailyBootOut": np.asarray(Rlist[1]), # Already an ndarray

      "dayPct_flux": R2pd(Rlist[2][0]),
      "dayPct_conc": R2pd(Rlist[2][1]),

      "ContConc": R2pd(Rlist[3]),
      "error": R2pd(Rlist[4]).loc["1"] # Converts from df to series
  }
  # Convert the date of all the outputs from number of days to an actual date
  for key in ["eList_Daily", "eList_Sample", "dayPct_flux", "dayPct_conc", "ContConc"]:
    repacked[key]["Date"] = pd.to_datetime(repacked[key]["Date"], unit="D")
    repacked[key].set_index("Date", inplace=True)

  return repacked


In [11]:
chem_dfs["Palena"].dtypes

Date              object
Time              object
Tripl             object
Mixing           float64
Qinst            float64
Qmean            float64
Low_Tide          object
Temp             float64
DO               float64
DO.1             float64
SPC              float64
Turb             float64
TSS              float64
TSS_sd           float64
dSi              float64
dSi_sd           float64
pH               float64
pH_sd             object
Alkalinity       float64
Alkalinity_sd     object
Na               float64
K                float64
Mg               float64
Ca               float64
F                 object
Cl               float64
SO4              float64
NH4               object
NOX              float64
DON               object
TN               float64
PO4              float64
DOP              float64
TP               float64
Fe (uM)          float64
Fe               float64
As               float64
Al               float64
DOC              float64
DON.1            float64


In [12]:
# seeds = pd.DataFrame(columns=rivers,
#                      index=chemicals,
#                      data=np.random.randint(100, size=(len(chemicals), len(rivers)))
#                      )
seeds = pd.read_csv(data_path/"copy_of_seeds.csv").set_index("chemicals")

In [13]:
on_unix = True # on_unix=False is untested

out_q = mp.Manager().Queue() # Fixed thanks to https://stackoverflow.com/a/56118981
done_q = mp.Manager().Queue()

def run_wrtds(args):
    riv, chem, which_flow, set_seed, nboot, nkalman = args
    display(riv, chem, which_flow)

    wdf = format_wdf(riv, chem, scaflow, chem_dfs, detection_limits, areas)
    info_df = format_infodf(riv, chem, areas)
    if which_flow == "unscaflow":
      flow_df = unscaflow[riv].rename("Q").reset_index()
    elif which_flow == "scaflow":
      flow_df = scaflow[riv].rename("Q").reset_index()
    else:
      raise ValueError("which_flow must be 'unscaflow' or 'scaflow'")

    eList = create_eList(pd2R(info_df), pd2R(flow_df), pd2R(wdf))
    Rout = repack_tables(estimate_tables(riv, chem, eList, nboot, nkalman, int(set_seed)))

    Rout["sample"] = wdf.set_index("Date")[chem]

    # Basically a return statement
    # if on_unix:
    #   all_daily_results.loc[{"river": riv, "chem": chem}] = Rout["ContConc"].to_xarray()
    #   all_dailyboot.loc[{"river": riv, "chem": chem}] = Rout["dailyBootOut"]
    #   all_dayPctConc.loc[{"river": riv, "chem": chem}] = Rout["dayPct_conc"]
    #   all_errorstats.loc[{"river": riv, "chem": chem}] = Rout["error"]
    #   all_samples.loc[{"river": riv, "chem": chem}] = wdf.set_index("Date")[chem].reindex(date_idx) # This is weird... use eList_Sample instead?
    # else:
    out_q.put((riv, chem, which_flow, Rout))

    print("Process done")
    done_q.put(riv, chem, which_flow)
    #f.close()

    # According to https://stackoverflow.com/a/17786444, multiprocessing on unix systems already uses shared memory on global variables...

    # TODO: Refactor so it uses yield and all that? show off?

In [15]:
seeds

,Aysen,Cisnes,Palena,Puelo,Yelcho
chemicals,,,,,
dSi,0,0,0,0,0
TN,0,0,0,0,0
TP,0,0,0,0,0
DOC,53,99,0,0,0
Fe,0,5,0,0,0
TSS,0,0,0,0,0


In [ ]:
run_wrtds(("Cisnes", "Fe", "scaflow", seeds.loc["Fe","Cisnes"], nboot, nkalman))

In [ ]:
date_idx

# 